In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import re

#해당 디렉토리 내부에 있는 모든 디렉토리의 이름을 가져온다
def get_dir_list(path):
    dir_list = []
    for root, dirs, files in os.walk(path):
        for dir in dirs:
            dir_list.append(dir)
    return dir_list

def get_file_list(dir_path):
    file_list = []
    for root, dirs, files in os.walk(dir_path):
        for file in files:
            file_list.append(file)
    return file_list

def parse_performance(log_str):
    results = {}
    sections = ['Validation', 'Test']

    for section in sections:
        pattern = rf"{section} - Time: ([\d.]+).*?Recall@20: ([\d.]+), NDCG@20: ([\d.]+);.*?Recall@50: ([\d.]+), NDCG@50: ([\d.]+);"
        match = re.search(pattern, log_str, re.DOTALL)
        if match:
            results[section.lower()] = {
                'recall@20': float(match.group(2)),
                'ndcg@20': float(match.group(3)),
                'recall@50': float(match.group(4)),
                'ndcg@50': float(match.group(5))
            }
        else:
            results[section.lower()] = {
                'recall@20': 0,
                'ndcg@20': 0,
                'recall@50': 0,
                'ndcg@50': 0
            }
    return results

def analyze_results(dir_path):
    file_list = get_file_list(dir_path)
    log_list = []
    for file in file_list:
        file_path = dir_path + '/' + file
        with open(file_path) as f:
            logs = "".join(f.readlines()[-9:])
        parsed = parse_performance(logs)
        log_list.append(parsed)
    return log_list
        

In [2]:
target_dataset = "Yelp"
target_model = "LightGCN"
target_method = "VQ"
target_dir = f"../log/{target_dataset}/{target_model}/{target_method}/"
dir_list = get_dir_list(target_dir)

max_valid_recall_20 = 0
best_dir = ""
for dir_name in dir_list:
    dir_path = os.path.join(target_dir, dir_name)
    log_list = analyze_results(dir_path)
    mean_valid_recall_20 = np.mean([log['validation']['recall@20'] for log in log_list])
    if mean_valid_recall_20 > max_valid_recall_20:
        max_valid_recall_20 = mean_valid_recall_20
        best_dir = dir_name

print(f"Target_dataset: {target_dataset}")
print(f"Target_model: {target_model}")
print(f"Target_method: {target_method}")
print(f"Best directory: {best_dir}")
print(f"Max Validation Recall@20: {max_valid_recall_20}")
test_recall20 = []
test_recall50 = []
test_ndcg20 = []
test_ndcg50 = []
for log in analyze_results(os.path.join(target_dir, best_dir)):
    test_recall20.append(log['test']['recall@20'])
    test_recall50.append(log['test']['recall@50'])
    test_ndcg20.append(log['test']['ndcg@20'])
    test_ndcg50.append(log['test']['ndcg@50'])
print(f"Test Recall@20: {np.mean(test_recall20):.4f}")
print(f"Test Recall@50: {np.mean(test_recall50):.4f}")
print(f"Test NDCG@20: {np.mean(test_ndcg20):.4f}")
print(f"Test NDCG@50: {np.mean(test_ndcg50):.4f}")







Target_dataset: Yelp
Target_model: LightGCN
Target_method: VQ
Best directory: 
Max Validation Recall@20: 0


FileNotFoundError: [Errno 2] No such file or directory: '../log/Yelp/LightGCN/VQ//log_0.0_2024.txt'